## CSCE 676 :: Data Mining and Analysis :: Texas A&M University :: Spring 2026


# Weekly Homework 9: Distributed Computing and Text Embeddings with Spark


***Goals of this homework:***
- Understand and apply core Spark concepts (RDDs, DataFrames, transformations, actions) to real-world data.
- Trace the MapReduce programming model through hands-on computation on a political network.
- Implement and interpret word2vec training data construction and the softmax scoring function.
- Reason about the tradeoffs between distributed computing tools and embedding-based text representations.


***Submission instructions:***

You should post your notebook to Canvas (look for the assignment there). Please name your submission **your-uin_hw9.ipynb**. Your notebook should be fully executed when you submit ... so run all the cells for us so we can see the output, then submit that.

***Grading philosophy:***

We are grading reasoning, judgment, and clarity, not just correctness. Show us that you understand the data, the constraints, and the limits of your conclusions.

***For each question, you need to respond with 2 cells:***
1. **[A Code Cell] Your Code:**
  - If code is not applicable for the question, you can skip this cell.
  - For tests: tests can be simple assertions or checks (e.g., using `assert` or `print` or small functions or visual inspection); formal testing frameworks are not required.
2. **[A Markdown Cell] Your Answer:** Write up your answers and explain them in complete sentences. Include any videos in this section as well; for videos, upload them to your TAMU Google Drive, and ensure they are set to be visible by the instruction team (set to: **anyone with a TAMU email can view**), then share the link to the video in the cell.

***At the end of each Section (A/B/C/...) include a cell for your resources:***

**[A Markdown Cell] Your Resources:** You need to cite 3 types of resources and note how they helped you: (1) Collaborators, (2) Web Sources (e.g. StackOverflow), and (3) AI Tools (you must also describe how you prompted, but we do not require any links to any specific chats). Specifically, use the following format as a template:
```
On my honor, I declare the following resources:
1. Collaborators:
- Reveille A.: Helped me understand that a df in pandas is a data structure kinda like a CSV.
- Sully A.: Helped me fix a bug with the vector addition of 2 columns.
- ...

2. Web Sources:
- https://stackoverflow.com/questions/46562479/python-pandas-data-frame-creation: how to create a pd df
- ...

3. AI Tools:
- ChatGPT: I gave it the homework .ipynb file and the congress edgelist, and told it to generate the code for the word2vec question, but it used the wrong window size, so I re-prompted it to use window_size=2 and that one was correct
- ...
```
***Why do we require this cell?*** This cell is important...

1. For academic integrity, you must give credit where credit is due.

2. We want you to pay attention to how you can successfully get help to move through problems! Is there someone you work with or an AI tool that helps you learn the material better? That's great! The point of engineering is to use your tools to solve hard problems, and part of graduate school is learning about how *you* learn and solve problems best.

***A reminder: you get out of it what you put into it.***
Do your best on these homeworks, show us your creativity, and ask for help when you need it -- good luck!

---
## Preliminaries: Setup

Run the cells below once before starting. They install dependencies, download the dataset, and start a Spark session. You do not need to modify these cells.

**Dataset:** The [Congress Twitter Network](https://snap.stanford.edu/data/congress-twitter.html) from SNAP. Each node is a member of the 117th US Congress. A directed edge A → B means A **follows** B on Twitter. Edge weights represent estimated information transmission probability between members.

In [ ]:
# ── Setup: Java + PySpark + graphframes ───────────────────────────────────────────────
!apt-get update -qq
!apt-get install -y openjdk-17-jdk-headless -qq
# Force PySpark 3.5 — graphframes has no Spark 4.0 JAR yet
!pip install pyspark==3.5.5 graphframes -q

import subprocess, os
java_home = subprocess.check_output(
    "dirname $(dirname $(readlink -f $(which java)))", shell=True
).decode().strip()
os.environ["JAVA_HOME"] = java_home
print("JAVA_HOME:", java_home)

# Download graphframes JAR for Spark 3.5 / Scala 2.12
import pyspark
jar_dir  = os.path.join(os.path.dirname(pyspark.__file__), 'jars')
jar_path = os.path.join(jar_dir, 'graphframes-0.8.4-spark3.5-s_2.12.jar')
jar_url  = ('https://repos.spark-packages.org/graphframes/graphframes/'
            '0.8.4-spark3.5-s_2.12/graphframes-0.8.4-spark3.5-s_2.12.jar')
if not os.path.exists(jar_path):
    subprocess.run(['curl', '-L', '-o', jar_path, jar_url], check=True)
    print('Downloaded graphframes JAR')
else:
    print('graphframes JAR already present')

In [ ]:
# ── Download dataset from SNAP ──────────────────────────────────────────────────────
!curl -L -o congress_network.zip https://snap.stanford.edu/data/congress_network.zip
!unzip -q congress_network.zip
!ls congress_network/

In [ ]:
# ── Imports ─────────────────────────────────────────────────────────────────────
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import pyspark
import pyspark.sql.functions as F
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark import SparkContext, SparkConf

In [ ]:
# ── Start Spark ───────────────────────────────────────────────────────────────────
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master('local[*]') \
    .config('spark.ui.port', '4050') \
    .config('spark.jars', jar_path) \
    .getOrCreate()

sc = spark.sparkContext
print('Spark version:', spark.version)  # should say 3.5.x

In [ ]:
# ── Load congress members ───────────────────────────────────────────────────────
with open('congress_network/congress_network_data.json') as f:
    data = json.load(f)[0]

usernames = data['usernameList']
print(f"Number of congress members: {len(usernames)}")
print(f"Sample usernames: {usernames[:5]}")

congress_members = spark.createDataFrame(
    [(str(i), usernames[i]) for i in range(len(usernames))],
    ["userid", "screen_name"]
)
congress_members.show(5)
print("Number of congress members tracked:", congress_members.count())

In [ ]:
# ── Load edges from edgelist ─────────────────────────────────────────────────────
edges_raw = []
with open('congress_network/congress.edgelist') as f:
    for line in f:
        parts = line.strip().split(' ', 2)
        if len(parts) >= 2:
            src, dst = parts[0], parts[1]
            weight = float(parts[2].split(':')[1].rstrip('}')) if len(parts) == 3 else 0.0
            edges_raw.append((src, dst, weight))

edges    = spark.createDataFrame(edges_raw, ["src", "dst", "weight"]).cache()
vertices = edges.select(F.col("src").alias("id")).union(edges.select("dst")).distinct()

edges.show(5)
print("# of edges:",    edges.count())
print("# of vertices:", vertices.count())

---
# A [63pts].

**Rubric**

[9 pts] Strong/Professional: Correct and complete implementation of the task; Reasonable assumptions, stated or implied, and justified; Thoughtful handling of real-world data issues (missingness, noise, scale, edge cases); Clear, concise explanations of what was done and why; Code is clean, readable, and well-structured, uses appropriate Spark and NumPy idioms, and would plausibly pass a professional code review; Interpretation goes beyond restating the output — draws a meaningful conclusion.

[5 pts] Partial/Developing: Core task mostly completed but with gaps, weak assumptions, or minor mistakes; Reasoning is shallow or mostly descriptive; Code works but is messy, repetitive, or fragile.

[0 pts] Minimal/Incorrect: Task is largely incorrect, missing, or misunderstands the goal; Little to no reasoning or justification; Code does not run or ignores constraints.

# 1. Exploratory Data Analysis with Spark DataFrames
Spark DataFrames are the high-level API introduced in recent versions of Spark that support SQL-style querying and are generally preferred over raw RDDs for structured data. Using the Spark DataFrames loaded in the Preliminaries, answer the following:

- How many unique users (nodes) and directed edges are in the network?
- Is **GOPLeader** in this dataset? If so, how many congress members follow them (i.e., how many incoming edges do they have)?
- Who are the top-5 most followed congress members by raw in-degree (number of incoming edges)? Join with `congress_members` to show screen names, not just node IDs.

*your answer here*

# 2. MapReduce In-Degree via Spark RDDs
In lecture we covered the MapReduce programming model: a **Map** step emits key-value pairs, a **group-by-key** (shuffle) step collects all values under each key, and a **Reduce** step aggregates them. Spark RDDs expose these operations directly.

Using `edges.rdd` (not DataFrames), compute the **raw in-degree** for every node in the network by following this explicit MapReduce pattern:
1. **Map**: for each edge `(src, dst, weight)`, emit `(dst, 1)`.
2. **Group / Reduce**: sum the 1s to get the in-degree for each destination node.

Show the top-10 nodes by in-degree (with screen names).

Then briefly compare the RDD approach to the DataFrame approach you used in Q1. What Spark abstractions are you relying on in each case, and which do you find more readable?

*your answer here*

# 3. Weighted In-Degree
Edge weights in this dataset represent the estimated **information transmission probability** between two congress members — not just whether a follow relationship exists, but how much information is likely to flow.

Using Spark DataFrames:
- Compute the **weighted in-degree** for each node (sum of incoming edge weights).
- Show the top-5 congress members by weighted in-degree (with screen names).
- Do the top-5 by weighted in-degree match the top-5 by raw in-degree from Q1? Explain what a mismatch would tell us about how information actually flows in this network.

*your answer here*

# 4. PageRank and Degree Analysis
PageRank is one of the most celebrated graph algorithms, originally designed by Larry Page and Sergey Brin to rank web pages. It models a random surfer: at each step, follow a random outgoing link with probability α (the damping factor), or teleport to a random node with probability 1−α.

- Build a **directed** NetworkX graph from the Spark edges (`.toPandas()` is fine here). Run PageRank with `alpha=0.85` and `max_iter=30`.
- Show the top-10 congress members by PageRank score (with screen names).
- Also show the top-5 by raw in-degree and top-5 by raw out-degree.
- Compare the PageRank ranking to the raw in-degree ranking from Q1 and the RDD-computed ranking from Q2. Are they the same? What does PageRank capture that raw in-degree does not? What does it mean for a congress member to have high out-degree but low PageRank?

*your answer here*

# 5. word2vec: Generating Training Data
In lecture we covered how word2vec builds its training set: for every word in a corpus, slide a window of size *m* and emit **(center word, context word)** pairs for all words within the window.

- Using the list of congress member screen names (`usernames`) as your corpus — treat the full list as a single "sentence" of words — write a function `generate_pairs(words, window_size)` that returns all `(center, context)` training pairs.
- Run it with `window_size=2` and print the first 15 pairs.
- How many total training pairs are generated from this corpus? Show the count.
- In the word2vec framework, each pair `(center, context)` contributes a term `p(context | center)` to the training objective. In one sentence, what does maximizing `p(context | center)` mean geometrically in terms of the embedding vectors?

*your answer here*

# 6. Softmax and Dot-Product Scoring
In lecture we traced through the full word2vec scoring pipeline for the training tuple **(cat, scratches)**:
1. Look up the center word embedding **v_c** from W_in.
2. Compute the dot product of **v_c** with every context word embedding **u_w** from W_out to get raw scores.
3. Apply **softmax** to turn those scores into a probability distribution.
4. The entry for *scratches* is interpreted as p(scratches | cat).

Implement this pipeline from scratch using NumPy:

- Define the following toy embeddings (these match the lecture slides):
  - `v_cat    = [0.2, 0.4, 0.9, -0.5, 0.4]`   ← center word embedding for "cat"
  - `u_scratches = [0.5, 0.1, 0.4, 0.2, 0.1]`  ← context word embedding for "scratches"
  - `u_the = [-0.3, 0.6, 0.2, 0.8, -0.1]`
  - `u_my  = [0.7, -0.2, 0.3, 0.1, 0.5]`
  - `u_frankie = [0.1, 0.3, -0.4, 0.6, 0.2]`

- Compute the raw dot-product score between `v_cat` and each context embedding. Verify that `v_cat · u_scratches = 0.44` (as shown in lecture, up to rounding).
- Write a `softmax(scores)` function and apply it to all four dot-product scores to get a probability distribution.
- Report p(scratches | cat). Is it higher than the other three probabilities? What would you expect after many training steps?

*your answer here*

# 7. Anomaly Detection in the Congress Network
The graph features you've computed so far — in-degree, out-degree, weighted in-degree, and PageRank — describe each congress member's structural role in the Twitter network. Most members will have "typical" profiles, but a few may stand out as structural outliers: unusually high PageRank relative to their raw in-degree, extreme follower imbalance, or very high weighted influence despite few connections.

Use **Isolation Forest** to surface these outliers:

1. Build a feature DataFrame with one row per congress member and four columns: `in_degree`, `out_degree`, `weighted_in_degree`, `pagerank`. (Collect the results from Q2–Q4 into pandas; nodes missing from any feature should be filled with 0.)
2. Standardize the features to zero mean and unit variance using `StandardScaler` before fitting.
3. Fit `IsolationForest(contamination=0.05, random_state=42)` and extract the raw `decision_function` score for each member (lower = more anomalous).
4. Show the **top-10 most anomalous** members (lowest scores) with their feature values.
5. For **at least 2** of the flagged members, explain which feature(s) make them stand out and what that pattern might mean in context.

*your answer here*

# Resources for Section A

```
On my honor, I declare the following resources:
1. Collaborators:
- ...

2. Web Sources:
- ...

3. AI Tools:
- ...
```

---
# B [30pts]. Interview Questions

We now pretend this is a real job interview. Here's some guidance on how to answer these questions:

1. Briefly restate the question and state any assumptions you are making.

2. Explain your reasoning out loud, focusing on tradeoffs, limitations, and constraints.

3. As a principle, keep your answers as short and clear as they can be (while still answering the question).

4. Write/speak in a conversational but professional tone (avoid being overly formal). For speaking: speak at a reasonable pace and volume, speak clearly, pause when you need to, and practice making "eye contact" with the camera. Keep a confident, positive, and professional tone. *For additional coaching and practice, the University Writing Center provides individual appointments: https://writingcenter.tamu.edu/make-an-appointment.*

There may not be a single correct answer. We are grading whether your reasoning is reasonable and aware of limitations.


**Rubric**

[10pt] Clear understanding of the question; reasonable assumptions; thoughtful reasoning that acknowledges tradeoffs and limitations; clear, concise communication in a conversational but professional tone (for speaking: clear pace, volume, and articulation).

[5pt] Basic understanding but shallow reasoning or unclear assumptions; communication is somewhat unclear, overly verbose, or overly informal/formal.

[0pt] Minimal, unclear, or incorrect response; poor communication or unprofessional tone.

# 1.
In lecture, we discussed **two major limitations of MapReduce** that motivated the design of Spark. What are those two limitations? How does Spark address each one? Use a concrete scenario from the Congress network analysis in Section A to illustrate when each of Spark's advantages would actually matter.

*your answer here*

# 2.
In word2vec, every word in the vocabulary gets **two** embeddings: one in W_in (used when the word is a center word) and one in W_out (used when the word is a context word). Why does the model need two separate matrices instead of just one? After training is complete, practitioners typically either average the two embeddings or simply discard W_out and use only W_in. Why is discarding W_out a reasonable choice, and when might averaging be better?

*your answer here*

# 3.
*(Video)* Walk through the word2vec training process using the sentence **"Frankie the cat scratches my arm"** with window size 2. Your walkthrough must cover:

1. All (center, context) training pairs generated when **"cat"** is the center word.
2. How p(scratches | cat) is computed step-by-step from W_in and W_out (dot product → softmax).
3. What the **error** is for the tuple (cat, scratches), and intuitively what gradient descent does to the embeddings to reduce that error.

*your answer here (include video link)*

# Resources for Section B

```
On my honor, I declare the following resources:
1. Collaborators:
- ...

2. Web Sources:
- ...

3. AI Tools:
- ...
```

---
# C [7pts]. What new questions do you have?
We want you to think bigger! Tell us what questions and curiosity this homework brings up for you.

**Rubric**

[7pt] Complete, thoughtful response.

[4pt] Partial response.

[0pt] Minimal response.

# 1.
What new questions do you have after this homework? Or, what topics are you curious about now? List at least 3.

*your answer here*